# SHAP explanations of the multi-class BDT

Exact TreeSHAP values from XGBoost itself (`booster.predict(..., pred_contribs=True)`), no `shap` package needed.

**Conventions used throughout**

* For every event, XGBoost gives one SHAP vector per class: $\phi_{k,f}$ for class $k$ and feature $f$, plus a bias term.
  They add up exactly to that class's raw score (margin) $m_k$, and $p_k = \mathrm{softmax}(m)_k$.
* Softmax doesn't change when the same constant is added to every margin, so a single class's raw SHAP values are partly
  arbitrary. The per-class plots therefore use **centered** SHAP values,
  $\tilde\phi_{k,f} = \phi_{k,f} - \frac{1}{K}\sum_j \phi_{j,f}$, which add up to $\tilde m_k = m_k - \overline{m}$
  (the log-probability of class $k$ relative to the geometric mean over classes).
* **Pairwise contrasts** $m_a - m_b = \log(p_a/p_b)$ are exactly additive too: their SHAP values are $\phi_a - \phi_b$.
  These are the cleanest way to ask "which variables separate signal class $a$ from background class $b$".
* SHAP values are in log-odds (natural log) units, before the class-prior correction (the BDT was trained with unit weights).
  The prior correction only adds a constant per class, so it shifts the bias term and doesn't change any feature's SHAP value.
* The events are a **class-balanced** random sample of BDT *test* events (the same events `train.py` used for testing),
  not the physics-weighted mix: each class contributes equally to the averages.

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

import numpy as np
import polars as pl
import xgboost as xgb
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from sklearn.metrics import roc_auc_score

from src.file_locations import intermediate_files_location
from src.signal_categories import train_category_labels, train_category_labels_latex, train_category_colors
from bdt_training_setup import get_training_vars, add_bdt_role_columns, select_training_events, bdt_feature_matrix

print(f"xgboost version: {xgb.__version__}")

In [ ]:
training = "all_vars_2026_11_04"

n_per_class = 1500        # BDT test events sampled per truth category
n_ncdelta_per_class = 1500 # true NC Delta radiative events (ncdeltarad_overlay, never trained on) per 1gamma category
seed = 42

training_dir = f"../training_outputs/{training}"
out_dir = f"{training_dir}/shap"
os.makedirs(out_dir, exist_ok=True)

with open(f"{training_dir}/training_config.json") as f:
    config = json.load(f)
assert config["signal_categories"] == "del1g_simple", config["signal_categories"]
training_vars = get_training_vars(config["training_vars"])
best_iteration = config["best_iteration"]

labels = train_category_labels
labels_latex = train_category_labels_latex
colors = train_category_colors
n_classes = len(labels)
n_features = len(training_vars)
cat = {label: i for i, label in enumerate(labels)}
print(f"{n_classes=}, {n_features=}, {best_iteration=}")

plt.rcParams.update({"font.size": 12})

def savefig(name):
    plt.savefig(f"{out_dir}/{name}.png", dpi=150, bbox_inches="tight")
    plt.savefig(f"{out_dir}/{name}.pdf", bbox_inches="tight")

# Event sample

In [ ]:
parquet_path = f"{intermediate_files_location}/presel_df_train_vars.parquet"
schema_names = pl.read_parquet_schema(parquet_path).keys()
key_cols = [c for c in ["filename", "filetype", "run", "subrun", "event"] if c in schema_names]
meta_cols = list(dict.fromkeys(key_cols + [
    "train_test_score", "wc_truth_NCDeltaRad", "wc_truth_numuCCDeltaRad", "wc_kine_reco_Enu",
    "coherent_1g_keep", "del1g_simple_signal_category", "wc_net_weight_open_data",
]))

meta = add_bdt_role_columns(
    pl.scan_parquet(parquet_path).with_row_index("row_nr").select(["row_nr"] + meta_cols).collect())

# exactly the events train.py tested on
test_meta = select_training_events(meta, config["signal_categories"]).filter(pl.col("bdt_role") == "test")

rng = np.random.default_rng(seed)
chosen = []
for k, label in enumerate(labels):
    rows = test_meta.filter(pl.col("del1g_simple_signal_category") == k)["row_nr"].to_numpy()
    chosen.append(rng.choice(rows, size=min(n_per_class, len(rows)), replace=False))
    print(f"{label:20s} test events: {len(rows):8d}, sampled: {len(chosen[-1])}")

# true NC Delta radiative events from the dedicated sample (bdt_role not_trained), for the signal-sample cross-check
ncdelta_meta = meta.filter((pl.col("filetype") == "ncdeltarad_overlay") & pl.col("wc_truth_NCDeltaRad")
                           & (pl.col("wc_kine_reco_Enu") > 0))
for k in [cat["1gNp"], cat["1g0p"]]:
    rows = ncdelta_meta.filter(pl.col("del1g_simple_signal_category") == k)["row_nr"].to_numpy()
    chosen.append(rng.choice(rows, size=min(n_ncdelta_per_class, len(rows)), replace=False))
    print(f"NCDeltaRad {labels[k]:9s} events: {len(rows):8d}, sampled: {len(chosen[-1])}")

sel_rows = np.unique(np.concatenate(chosen))
print(f"loading {len(sel_rows)} events x {n_features} features...")
t0 = time.time()
sample_df = (
    pl.scan_parquet(parquet_path).with_row_index("row_nr")
    .filter(pl.col("row_nr").is_in(sel_rows))
    .select(["row_nr"] + training_vars)
    .collect()
    .join(meta.select(["row_nr", "bdt_role"] + meta_cols), on="row_nr", how="left")
    .sort("row_nr")
)
print(f"loaded in {time.time() - t0:.0f} s, {sample_df.shape=}")

y = sample_df["del1g_simple_signal_category"].to_numpy()
filetype = sample_df["filetype"].to_numpy()
is_ncdelta = filetype == "ncdeltarad_overlay"
is_main = sample_df["bdt_role"].to_numpy() == "test"
assert not np.any(is_ncdelta & is_main)

# SHAP values

In [ ]:
booster = xgb.Booster()
booster.load_model(f"{training_dir}/bdt.json")
iteration_range = (0, best_iteration + 1) # the saved booster keeps the 50 rounds after the best iteration too

X = bdt_feature_matrix(sample_df, training_vars) # same missing-value preprocessing as training
n_events = len(X)
phi = np.empty((n_events, n_classes, n_features + 1), dtype=np.float32) # last entry: bias
t0 = time.time()
chunk = 2000
for start in range(0, n_events, chunk):
    phi[start:start + chunk] = booster.predict(
        xgb.DMatrix(X[start:start + chunk], missing=np.nan), pred_contribs=True, iteration_range=iteration_range)
print(f"SHAP for {n_events} events in {time.time() - t0:.0f} s, {phi.shape=}, {phi.nbytes / 1e9:.2f} GB")

margin = phi.sum(axis=2, dtype=np.float64)
probs = np.exp(margin - margin.max(axis=1, keepdims=True))
probs /= probs.sum(axis=1, keepdims=True)

# centered SHAP values (see the top of the notebook); features only, bias dropped
phi_c = phi[:, :, :-1] - phi[:, :, :-1].mean(axis=1, keepdims=True)

In [ ]:
# sanity check: the SHAP sums reproduce the probabilities train.py saved
pred = (pl.scan_parquet(f"{training_dir}/predictions.parquet")
        .select(key_cols + [f"prob_{label}" for label in labels])
        .join(sample_df.lazy().select(key_cols), on=key_cols, how="semi").collect())
check = sample_df.select(key_cols).join(pred, on=key_cols, how="left", maintain_order="left")
assert check.height == n_events, "duplicate keys in predictions.parquet"
stored = check.select([f"prob_{label}" for label in labels]).to_numpy()
print(f"max |SHAP-sum probability - stored probability| = {np.nanmax(np.abs(stored - probs)):.2e}, "
      f"events without a stored prediction: {np.isnan(stored).any(axis=1).sum()}")

In [ ]:
def framework(var):
    for prefix, name in [("wc_", "WC"), ("lantern_", "LANTERN"), ("glee_", "gLEE"), ("pandora_", "Pandora")]:
        if var.startswith(prefix):
            return name
    raise ValueError(f"no framework prefix: {var}")

frameworks = ["WC", "Pandora", "LANTERN", "gLEE"]
framework_colors = {"WC": "tab:blue", "Pandora": "tab:orange", "LANTERN": "tab:green", "gLEE": "tab:red"}
feature_framework = np.array([framework(v) for v in training_vars])
for fw in frameworks:
    print(f"{fw:9s}: {np.sum(feature_framework == fw)} features")

# per-event SHAP summed over each framework's features: exact, since SHAP is additive
group_phi_c = np.stack([phi_c[:, :, feature_framework == fw].sum(axis=2) for fw in frameworks], axis=2)

# Global importance

Mean |centered SHAP| over the class-balanced main sample, for each class's margin. The stacked bar shows how
much of each feature's total influence goes to each class.

In [ ]:
mean_abs = np.abs(phi_c[is_main]).mean(axis=0)  # (classes, features)
total_importance = mean_abs.sum(axis=0)
order = np.argsort(total_importance)[::-1]

top_n = 40
fig, ax = plt.subplots(figsize=(11, 14))
left = np.zeros(top_n)
idx = order[:top_n][::-1]
for k in range(n_classes):
    ax.barh(np.arange(top_n), mean_abs[k, idx], left=left, color=colors[k], label=labels_latex[k],
            edgecolor="none")
    left += mean_abs[k, idx]
ax.set_yticks(np.arange(top_n))
ax.set_yticklabels([training_vars[i] for i in idx], fontsize=9)
ax.set_xlabel(r"mean |centered SHAP| summed over classes (log-odds)")
ax.set_title(f"Top {top_n} features by SHAP importance")
ax.legend(fontsize=8, loc="lower right", ncol=2)
savefig("global_importance_stacked")
plt.show()

In [ ]:
# how concentrated is the BDT? cumulative share of total mean |SHAP| vs number of features, per class
fig, ax = plt.subplots(figsize=(9, 6))
for k in range(n_classes):
    s = np.sort(mean_abs[k])[::-1]
    ax.plot(np.arange(1, n_features + 1), np.cumsum(s) / s.sum(), color=colors[k], label=labels_latex[k], lw=1.5)
s = np.sort(total_importance)[::-1]
ax.plot(np.arange(1, n_features + 1), np.cumsum(s) / s.sum(), color="k", lw=3, label="all classes")
ax.set_xscale("log")
ax.set_xlabel("number of features (ranked by SHAP importance for that class)")
ax.set_ylabel("cumulative fraction of mean |SHAP|")
ax.axhline(0.9, color="gray", ls="--", lw=1)
ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=2)
savefig("cumulative_importance")
plt.show()
n90 = [int(np.searchsorted(np.cumsum(np.sort(mean_abs[k])[::-1]) / mean_abs[k].sum(), 0.9)) + 1 for k in range(n_classes)]
for k in range(n_classes):
    print(f"{labels[k]:20s} features for 90% of mean |SHAP|: {n90[k]}")

In [ ]:
# SHAP ranking vs XGBoost's built-in gain importance
fi = pl.read_csv(f"{training_dir}/feature_importances.csv")
gain = dict(zip(fi["feature"], fi["gain_importance"]))
gain_arr = np.array([gain.get(v, 0.0) for v in training_vars])
shap_rank = np.empty(n_features, int); shap_rank[order] = np.arange(1, n_features + 1)
gain_rank = np.empty(n_features, int); gain_rank[np.argsort(gain_arr)[::-1]] = np.arange(1, n_features + 1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(gain_rank, shap_rank, s=6, c=[framework_colors[fw] for fw in feature_framework], alpha=0.6)
for i in order[:15]:
    ax.annotate(training_vars[i], (gain_rank[i], shap_rank[i]), fontsize=7)
ax.set_xscale("log"); ax.set_yscale("log")
ax.plot([1, n_features], [1, n_features], "k--", lw=1)
ax.set_xlabel("gain importance rank"); ax.set_ylabel("SHAP importance rank")
for fw in frameworks:
    ax.scatter([], [], c=framework_colors[fw], label=fw)
ax.legend()
ax.set_title("SHAP vs gain feature ranking")
savefig("shap_vs_gain_rank")
plt.show()
print("top 20 by SHAP:   ", [training_vars[i] for i in order[:20]])
print("top 20 by gain:   ", [training_vars[i] for i in np.argsort(gain_arr)[::-1][:20]])

# Feature × class heatmaps

Left: mean |centered SHAP| of each class's margin over all sampled events (how much the feature moves that class's
score). Right: mean *signed* centered SHAP of class $k$'s margin over events that truly are class $k$ (positive = the
feature helps the BDT recognize its own events).

In [ ]:
top_n = 50
idx = order[:top_n]
fig, axes = plt.subplots(1, 2, figsize=(22, 17), sharey=True)

im = axes[0].imshow(mean_abs[:, idx].T, aspect="auto", cmap="viridis")
axes[0].set_title("mean |centered SHAP|, all events")
plt.colorbar(im, ax=axes[0], fraction=0.03, pad=0.01)

own = np.stack([phi_c[is_main & (y == k), k, :].mean(axis=0) for k in range(n_classes)])  # (classes, features)
vmax = np.abs(own[:, idx]).max()
im = axes[1].imshow(own[:, idx].T, aspect="auto", cmap="RdBu_r", norm=TwoSlopeNorm(0, -vmax, vmax))
axes[1].set_title("mean signed centered SHAP of class k, true class-k events")
plt.colorbar(im, ax=axes[1], fraction=0.03, pad=0.01)

for ax in axes:
    ax.set_xticks(np.arange(n_classes))
    ax.set_xticklabels(labels_latex, rotation=90, fontsize=10)
axes[0].set_yticks(np.arange(top_n))
axes[0].set_yticklabels([training_vars[i] for i in idx], fontsize=8)
plt.tight_layout()
savefig("feature_class_heatmap")
plt.show()

In [ ]:
# compact version: mean |centered SHAP| only, top 20 features
top_n = 20
idx = order[:top_n]
fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(mean_abs[:, idx].T, aspect="auto", cmap="viridis")
ax.set_xticks(np.arange(n_classes)); ax.set_xticklabels(labels_latex, rotation=90, fontsize=11)
ax.set_yticks(np.arange(top_n)); ax.set_yticklabels([training_vars[i] for i in idx], fontsize=10)
cb = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cb.set_label("mean |centered SHAP| (log-odds)")
ax.set_title(f"Top {top_n} features: mean |centered SHAP| per class")
plt.tight_layout()
savefig("feature_class_heatmap_abs_top20")
plt.show()

In [ ]:
# the same, column-normalized: which features dominate each class (each column sums to 1 over all features)
norm_abs = mean_abs / mean_abs.sum(axis=1, keepdims=True)
top_per_class = np.unique(np.concatenate([np.argsort(norm_abs[k])[::-1][:6] for k in range(n_classes)]))
top_per_class = top_per_class[np.argsort(total_importance[top_per_class])[::-1]]
fig, ax = plt.subplots(figsize=(13, 0.22 * len(top_per_class) + 3))
im = ax.imshow(norm_abs[:, top_per_class].T, aspect="auto", cmap="magma_r")
ax.set_xticks(np.arange(n_classes)); ax.set_xticklabels(labels_latex, rotation=90)
ax.set_yticks(np.arange(len(top_per_class)))
ax.set_yticklabels([training_vars[i] for i in top_per_class], fontsize=8)
plt.colorbar(im, ax=ax, label="fraction of class's total mean |SHAP|", fraction=0.03)
ax.set_title("Top 6 features of every class (fraction of that class's mean |SHAP|)")
savefig("feature_class_heatmap_top6_per_class")
plt.show()

# Importance by reconstruction framework

Per event, the centered SHAP values of all features from one framework are summed (exact, because SHAP is additive),
so correlated near-duplicate variables within a framework no longer split the credit.

In [ ]:
group_mean_abs = np.abs(group_phi_c[is_main]).mean(axis=0)  # (classes, frameworks)
fig, axes = plt.subplots(2, 1, figsize=(15, 11), sharex=True)
x = np.arange(n_classes)
width = 0.8 / len(frameworks)
for g, fw in enumerate(frameworks):
    axes[0].bar(x + (g - 1.5) * width, group_mean_abs[:, g], width, color=framework_colors[fw], label=fw)
axes[0].set_ylabel("mean |framework SHAP sum| (log-odds)")
axes[0].legend()
axes[0].set_title("Influence of each framework on each class's centered margin")

frac = group_mean_abs / group_mean_abs.sum(axis=1, keepdims=True)
bottom = np.zeros(n_classes)
for g, fw in enumerate(frameworks):
    axes[1].bar(x, frac[:, g], bottom=bottom, color=framework_colors[fw], label=fw)
    bottom += frac[:, g]
axes[1].set_ylabel("fraction")
axes[1].set_xticks(x); axes[1].set_xticklabels(labels_latex, rotation=90)
plt.tight_layout()
savefig("framework_importance_by_class")
plt.show()

# per-feature view: sum of per-feature mean |SHAP| by framework (counts correlated features separately)
per_feature = np.array([total_importance[feature_framework == fw].sum() for fw in frameworks])
per_event = np.array([np.abs(group_phi_c[is_main][:, :, g]).mean(axis=0).sum() for g in range(len(frameworks))])
for g, fw in enumerate(frameworks):
    print(f"{fw:9s}: sum of per-feature mean|SHAP| {per_feature[g]:7.2f}, mean|framework sum| {per_event[g]:7.2f} "
          f"(ratio {per_event[g] / per_feature[g]:.2f}: lower = more cancellation between that framework's features)")

In [ ]:
# distributions of each framework's contribution to the event's own (true-class) centered margin
fig, axes = plt.subplots(4, 5, figsize=(22, 16), sharey=True)
for k, ax in enumerate(axes.flat):
    sel = is_main & (y == k)
    data = [group_phi_c[sel, k, g] for g in range(len(frameworks))]
    # gaussian_kde fails on constant data
    data = [d + 1e-6 * np.random.default_rng(0).standard_normal(len(d)) if np.std(d) == 0 else d for d in data]
    parts = ax.violinplot(data, vert=False, showmedians=True, widths=0.85)
    for g, body in enumerate(parts["bodies"]):
        body.set_facecolor(framework_colors[frameworks[g]]); body.set_alpha(0.7)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(np.arange(1, len(frameworks) + 1)); ax.set_yticklabels(frameworks)
    ax.set_title(labels_latex[k], fontsize=11)
fig.supxlabel("framework SHAP sum for the event's own class (centered log-odds)")
plt.tight_layout()
savefig("framework_shap_violins_own_class")
plt.show()

# Beeswarm plots: every class

Each dot is an event; x is the centered SHAP value of that class's margin, color is the feature's value (percentile
rank among the sample, gray = missing). All sampled events are shown, not only those of the class.

In [ ]:
feature_pct = np.full(X.shape, np.nan, dtype=np.float32)
for f in range(n_features):
    col = X[:, f]
    ok = np.isfinite(col)
    if ok.sum() > 1:
        ranks = np.argsort(np.argsort(col[ok], kind="stable"), kind="stable")
        feature_pct[ok, f] = ranks / (ok.sum() - 1)

def beeswarm(ax, shap_vals, feat_idx, events, title, max_points=6000, rng_seed=0):
    # shap_vals: (n_events_total, n_features) array; rows events, columns feature_idx are plotted top to bottom
    rng_local = np.random.default_rng(rng_seed)
    ev = np.flatnonzero(events)
    if len(ev) > max_points:
        ev = rng_local.choice(ev, max_points, replace=False)
    n_rows = len(feat_idx)
    for row, f in enumerate(feat_idx):
        ypos = n_rows - 1 - row
        s = shap_vals[ev, f]
        counts, edges = np.histogram(s, bins=60)
        b = np.clip(np.digitize(s, edges) - 1, 0, len(counts) - 1)
        jitter = rng_local.uniform(-1, 1, len(s)) * 0.42 * counts[b] / counts.max()
        c = feature_pct[ev, f]
        miss = ~np.isfinite(c)
        ax.scatter(s[miss], ypos + jitter[miss], s=2, color="0.7", rasterized=True)
        sc = ax.scatter(s[~miss], ypos + jitter[~miss], s=2, c=c[~miss], cmap="coolwarm", vmin=0, vmax=1,
                        rasterized=True)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels([training_vars[f] for f in feat_idx][::-1], fontsize=8)
    ax.set_title(title, fontsize=11)
    return sc

n_show = 15
for k in range(n_classes):
    feat_idx = np.argsort(mean_abs[k])[::-1][:n_show]
    fig, ax = plt.subplots(figsize=(10, 7))
    sc = beeswarm(ax, phi_c[:, k, :], feat_idx, is_main, f"{labels_latex[k]}: centered margin")
    ax.set_xlabel(f"centered SHAP for class {labels[k]} (log-odds)")
    cb = plt.colorbar(sc, ax=ax, fraction=0.03); cb.set_label("feature value (percentile; gray = missing)")
    savefig(f"beeswarm_class_{k:02d}_{labels[k]}")
    plt.show()

In [ ]:
# all 20 classes on one page: top 3 features per class, feature names written above each row
n_show = 3
fig, axes = plt.subplots(4, 5, figsize=(26, 17))
for k, ax in enumerate(axes.flat):
    feat_idx = np.argsort(mean_abs[k])[::-1][:n_show]
    sc = beeswarm(ax, phi_c[:, k, :], feat_idx, is_main, labels_latex[k], max_points=4000)
    ax.set_yticks([])
    for row, f in enumerate(feat_idx):
        ax.text(0.01, n_show - 1 - row + 0.47, training_vars[f], transform=ax.get_yaxis_transform(),
                ha="left", va="top", fontsize=9)
    ax.set_ylim(-0.55, n_show - 0.5 + 0.1)
    ax.set_title(labels_latex[k], fontsize=14)
    ax.tick_params(axis="x", labelsize=10)
fig.supxlabel("centered SHAP for the panel's class (log-odds)", fontsize=15)
fig.tight_layout(rect=(0, 0, 0.94, 1))
cax = fig.add_axes([0.95, 0.25, 0.012, 0.5])
cb = fig.colorbar(sc, cax=cax); cb.set_label("feature value (percentile; gray = missing)", fontsize=13)
savefig("beeswarm_grid_top3")
plt.show()

# Signal vs background contrasts

SHAP of $m_a - m_b = \log(p_a/p_b)$, over the true class-$a$ and class-$b$ events only. For each pair: a beeswarm,
dependence plots of the top features, the per-framework contributions, and how well each framework's SHAP sum alone
separates the two classes (ROC AUC).

In [ ]:
contrasts = [
    ("1gNp", "NC1pi0_Np"),
    ("1g0p", "NC1pi0_0p"),
    ("1gNp1mu", "numuCC1pi0_Np"),
    ("1g0p1mu", "numuCC1pi0_0p"),
    ("1gNp", "nueCC_Np"),
    ("1g0p", "nueCC_0p"),
    ("1g_outFV", "1pi0_outFV"),
]

def contrast_shap(a, b):
    return phi[:, cat[a], :-1] - phi[:, cat[b], :-1]

In [ ]:
contrast_summary = {}
for a, b in contrasts:
    s = contrast_shap(a, b)
    events = is_main & ((y == cat[a]) | (y == cat[b]))
    imp = np.abs(s[events]).mean(axis=0)
    feat_idx = np.argsort(imp)[::-1][:10]
    contrast_summary[(a, b)] = (s, events, imp)

    fig, ax = plt.subplots(figsize=(11, 6))
    sc = beeswarm(ax, s, feat_idx, events,
                  f"{labels_latex[cat[a]]} vs {labels_latex[cat[b]]}: SHAP of $\\log(p_a/p_b)$")
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
    ax.set_xlabel("SHAP (log-odds)")
    cb = plt.colorbar(sc, ax=ax, fraction=0.03); cb.set_label("feature value (percentile; gray = missing)")
    plt.tight_layout()
    savefig(f"contrast_beeswarm_{a}_vs_{b}")
    plt.show()

In [ ]:
def dependence_plot(ax, f, s, events, a, b, rng_seed=0):
    rng_local = np.random.default_rng(rng_seed)
    x_all = X[:, f].copy()
    # sentinels that survive the +-1e10 missing-value cut (e.g. -2.1e9, -9999) go in the missing strip too
    x_all[(np.abs(x_all) >= 1e5) | (x_all <= -999)] = np.nan
    ok_events = events & np.isfinite(x_all)
    finite_x = x_all[ok_events]
    if len(finite_x) > 0:
        lo, hi = np.percentile(finite_x, [0.5, 99.5])
        if hi <= lo:
            lo, hi = lo - 0.5, hi + 0.5
    else:
        lo, hi = 0.0, 1.0
    span = hi - lo
    miss_x = lo - 0.08 * span  # missing values drawn in a strip to the left
    for cls in [b, a]:
        sel = events & (y == cat[cls])
        xv = x_all[sel].copy()
        miss = ~np.isfinite(xv)
        xv[miss] = miss_x + rng_local.uniform(-0.02, 0.02, miss.sum()) * span
        xv = np.clip(xv, lo - 0.1 * span, hi + 0.02 * span)
        ax.scatter(xv, s[sel, f], s=2, alpha=0.35, color=colors[cat[cls]], rasterized=True,
                   label=f"{labels_latex[cat[cls]]}")
    ax.axvline(lo - 0.04 * span, color="0.6", lw=0.8, ls=":")
    ax.axhline(0, color="k", lw=0.8)
    nan_frac = np.mean(~np.isfinite(x_all[events]))
    ax.set_title(f"{training_vars[f]}\n(missing/sentinel {100 * nan_frac:.0f}%: left strip)", fontsize=9)
    ax.set_xlim(lo - 0.12 * span, hi + 0.04 * span)

for a, b in contrasts:
    s, events, imp = contrast_summary[(a, b)]
    feat_idx = np.argsort(imp)[::-1][:12]
    fig, axes = plt.subplots(3, 4, figsize=(20, 13))
    for ax, f in zip(axes.flat, feat_idx):
        dependence_plot(ax, f, s, events, a, b)
    axes.flat[0].legend(markerscale=6, fontsize=9)
    fig.suptitle(f"Dependence plots: {labels_latex[cat[a]]} vs {labels_latex[cat[b]]} (SHAP of $\\log(p_a/p_b)$ vs feature value)")
    plt.tight_layout()
    savefig(f"contrast_dependence_{a}_vs_{b}")
    plt.show()

In [ ]:
# per-framework contributions to each contrast, and the separation power of each framework's SHAP sum alone
fig, axes = plt.subplots(len(contrasts), 1, figsize=(14, 3.3 * len(contrasts)))
auc_rows = []
for ax, (a, b) in zip(axes, contrasts):
    s, events, imp = contrast_summary[(a, b)]
    group_s = np.stack([s[:, feature_framework == fw].sum(axis=1) for fw in frameworks], axis=1)
    is_a = (y == cat[a])[events]
    total = s[events].sum(axis=1)
    row = {"contrast": f"{a} vs {b}", "all": roc_auc_score(is_a, total)}
    edges = np.linspace(np.percentile(group_s[events], 0.5), np.percentile(group_s[events], 99.5), 70)
    for g, fw in enumerate(frameworks):
        row[fw] = roc_auc_score(is_a, group_s[events][:, g])
        ax.hist(group_s[events][is_a, g], bins=edges, histtype="step", color=framework_colors[fw], lw=2,
                density=True, label=f"{fw}, true {a}")
        ax.hist(group_s[events][~is_a, g], bins=edges, histtype="step", color=framework_colors[fw], lw=1.2,
                ls="--", density=True, label=f"{fw}, true {b}")
    auc_rows.append(row)
    ax.set_yscale("log")
    ax.set_title(f"{labels_latex[cat[a]]} vs {labels_latex[cat[b]]}: framework SHAP sums (solid: true {a}, dashed: true {b})",
                 fontsize=11)
    ax.legend(fontsize=7, ncol=5)
axes[-1].set_xlabel("framework SHAP sum (log-odds)")
plt.tight_layout()
savefig("contrast_framework_distributions")
plt.show()

auc_df = pl.DataFrame(auc_rows)
with pl.Config(tbl_rows=50, float_precision=3):
    print("ROC AUC of the full log(p_a/p_b) and of each framework's SHAP contribution alone")
    print(auc_df)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(auc_df.select(["all"] + frameworks).to_numpy(), cmap="viridis", vmin=0.5, vmax=1, aspect="auto")
ax.set_xticks(np.arange(len(frameworks) + 1)); ax.set_xticklabels(["all"] + frameworks)
ax.set_yticks(np.arange(len(contrasts))); ax.set_yticklabels(auc_df["contrast"].to_list())
for i in range(len(contrasts)):
    for j, c in enumerate(["all"] + frameworks):
        ax.text(j, i, f"{auc_df[c][i]:.3f}", ha="center", va="center", color="w", fontsize=9)
plt.colorbar(im, ax=ax, label="ROC AUC")
ax.set_title("Separation from each framework's SHAP contribution alone")
savefig("contrast_framework_auc")
plt.show()

# Individual events (waterfall plots)

How the BDT reached its answer for single events: starting from the bias (the average $\log(p_a/p_b)$), each bar adds
one feature's SHAP value; the remaining features are lumped together.

In [ ]:
def waterfall(ax, i, a, b, n_top=14):
    s = phi[i, cat[a], :] - phi[i, cat[b], :]
    bias, contrib = s[-1], s[:-1]
    top = np.argsort(np.abs(contrib))[::-1][:n_top]
    rest = contrib.sum() - contrib[top].sum()
    names = ["bias"] + [f"{training_vars[f]} = {X[i, f]:.3g}" for f in top] + [f"{n_features - n_top} other features"]
    vals = np.concatenate([[bias], contrib[top], [rest]])
    start = np.concatenate([[0], np.cumsum(vals)[:-1]])
    ypos = np.arange(len(vals))[::-1]
    ax.barh(ypos, vals, left=start, color=["0.6"] + ["tab:red" if v > 0 else "tab:blue" for v in vals[1:]],
            edgecolor="k", lw=0.3)
    ax.axvline(s.sum(), color="k", ls="--", lw=1)
    ax.set_yticks(ypos); ax.set_yticklabels(names, fontsize=8)
    ax.set_title(f"true {labels_latex[y[i]]} ({filetype[i]}): $\\log(p_a/p_b)$ = {s.sum():.2f}, "
                 f"$p_{{{a}}}$ = {probs[i, cat[a]]:.3f}, $p_{{{b}}}$ = {probs[i, cat[b]]:.3f}", fontsize=9)
    ax.set_xlabel(f"log(p_{a} / p_{b})")

for a, b in [("1gNp", "NC1pi0_Np"), ("1g0p", "NC1pi0_0p"), ("1gNp1mu", "numuCC1pi0_Np")]:
    lo = margin[:, cat[a]] - margin[:, cat[b]]
    sig = np.flatnonzero(is_main & (y == cat[a]))
    bkg = np.flatnonzero(is_main & (y == cat[b]))
    examples = [
        ("most signal-like true signal", sig[np.argmax(lo[sig])]),
        ("median true signal", sig[np.argsort(lo[sig])[len(sig) // 2]]),
        ("most background-like true signal", sig[np.argmin(lo[sig])]),
        ("most signal-like true background", bkg[np.argmax(lo[bkg])]),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(22, 13))
    for ax, (desc, i) in zip(axes.flat, examples):
        waterfall(ax, i, a, b)
        ax.text(0.99, 0.02, desc, transform=ax.transAxes, ha="right", va="bottom", fontsize=11, weight="bold")
    plt.tight_layout()
    savefig(f"waterfalls_{a}_vs_{b}")
    plt.show()

# Cross-check: signal training samples vs true NC $\Delta$ radiative events

The 1$\gamma$ classes are trained on the del1g/iso1g samples. Real NC $\Delta$ radiative events
(`ncdeltarad_overlay`, never trained on) of the same truth category should be explained the same way. Large
differences in the SHAP patterns would point to features that distinguish the samples rather than the physics.

In [ ]:
signal_filetypes = ["delete_one_gamma_overlay", "isotropic_one_gamma_overlay"]
for k in [cat["1gNp"], cat["1g0p"]]:
    groups = {f"del1g, true {labels[k]}": is_main & (y == k) & (filetype == "delete_one_gamma_overlay"),
              f"iso1g, true {labels[k]}": is_main & (y == k) & (filetype == "isotropic_one_gamma_overlay"),
              f"NC$\\Delta$rad, true {labels[k]}": is_ncdelta & (y == k)}
    groups = {name: sel for name, sel in groups.items() if sel.sum() >= 50}
    for name, sel in groups.items():
        argmax_frac = np.mean(np.argmax(probs[sel], axis=1) == k)
        print(f"{name:35s} n={sel.sum():5d}  mean p={probs[sel, k].mean():.3f}  argmax=={labels[k]}: {argmax_frac:.3f}")

    fig, axes = plt.subplots(1, 3, figsize=(24, 8), gridspec_kw={"width_ratios": [1.2, 1, 2]})
    bins = np.linspace(0, 1, 41)
    group_colors = ["tab:orange", "tab:olive", "tab:purple"][-len(groups):] if len(groups) == 2 else ["tab:orange", "tab:olive", "tab:purple"]
    for (name, sel), col in zip(groups.items(), group_colors):
        axes[0].hist(probs[sel, k], bins=bins, histtype="step", lw=2, density=True, color=col, label=name)
    axes[0].set_yscale("log"); axes[0].set_xlabel(f"p({labels[k]})"); axes[0].legend(fontsize=9)
    axes[0].set_title(f"BDT score of the true class {labels_latex[k]}")

    x = np.arange(len(frameworks))
    w = 0.8 / len(groups)
    for j, ((name, sel), col) in enumerate(zip(groups.items(), group_colors)):
        axes[1].bar(x + (j - (len(groups) - 1) / 2) * w, group_phi_c[sel, k, :].mean(axis=0), w, color=col, label=name)
    axes[1].set_xticks(x); axes[1].set_xticklabels(frameworks)
    axes[1].axhline(0, color="k", lw=0.8)
    axes[1].set_ylabel(f"mean framework SHAP sum, centered margin of {labels[k]}")
    axes[1].legend(fontsize=9)

    sel_sig = is_main & (y == k) & np.isin(filetype, signal_filetypes) # del1g + iso1g as sampled
    sel_ncd = is_ncdelta & (y == k)
    diff = phi_c[sel_sig, k, :].mean(axis=0) - phi_c[sel_ncd, k, :].mean(axis=0)
    top = np.argsort(np.abs(diff))[::-1][:25][::-1]
    axes[2].barh(np.arange(len(top)), diff[top], color=[framework_colors[feature_framework[f]] for f in top])
    axes[2].set_yticks(np.arange(len(top))); axes[2].set_yticklabels([training_vars[f] for f in top], fontsize=8)
    axes[2].axvline(0, color="k", lw=0.8)
    axes[2].set_xlabel("mean SHAP (del1g + iso1g) - mean SHAP (NC$\\Delta$rad)")
    axes[2].set_title("Largest per-feature differences (color: framework)")
    plt.tight_layout()
    savefig(f"ncdelta_check_{labels[k]}")
    plt.show()

In [ ]:
print(f"saved plots to {os.path.abspath(out_dir)}")
print(sorted(os.listdir(out_dir)))